In [ ]:
from pathlib import Path
import sys
exp_dir = Path.cwd().resolve().parent
if str(exp_dir) not in sys.path:
    sys.path.insert(0, str(exp_dir))
from decoder_only import similarity_degree, similarity_degree_k, get_last_token_and_word_vector, get_last_aim_word_k_vector, get_top_k_attned_v, get_all_hidden_states, get_all_k_vectors
import pickle
import json
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
import tqdm
from torch import Tensor
from typing import List, Tuple, Dict
import numpy as np
from torch.nn import functional as F
out_data_dir = "../../data/output_data/icews_yago/decoder_only"
input_data_dir = "../../data/output_data/icews_yago"

#### 使用到的prompt模板

In [ ]:
sample_prompt1 = "Focus on entity deduplication and disambiguation in knowledge graph, by extracting the core discriminative semantics of the entity in its context. Entity: {word}, context: {relations}. The core semanitcs of the entity \" {word} \" in the current context is (keep the information that can distinguish the entity from other similar entities do not just output the generalized theme):"
sample_prompt2 = "The entity \" {word} \" of type \"{type}\" might have different meanings in different contexts (ambiguation), or might share the same essence of an entity with a different name (duplication). For example, in relationships: {relations}, the entity \" {word} \""

#### 从Node以及Relationship中提取signature的函数，signature是一个字符串，包含了节点的name、type以及其他属性（除了id相关的属性）。

In [ ]:
def get_node_signature(node: Node) ->str:
    properties = {}
    for k,v in node.properties.items():
        # 如果k中包含"ID"或"id"或"Id"或"iD"或“ISNI”，就跳过这个属性
        if "ID" in k or "id" in k or "Id" in k or "iD" in k or "ISNI" in k:
            continue
        properties[k] = v
    property_str = ",".join(f"{k}={v}" for k, v in properties.items() if k != "name")
    node_type = node.type if node.type != "unknown" else "Entity"
    signature = f"{node.properties["name"]}(type={node_type})"
    return signature

def get_relationship_signature(relationship: Relationship) -> str:
    node1_signature = get_node_signature(relationship.source)
    node2_signature = get_node_signature(relationship.target)
    # 关系属性加进去
    properties = {}
    for k,v in relationship.properties.items():
        # 如果k中包含"ID"或"id"或"Id"或"iD"或“ISNI”，就跳过这个属性
        if "ID" in k or "id" in k or "Id" in k or "iD" in k or "ISNI" in k:
            continue
        properties[k] = v
    property_str = ",".join(f"{k}={v}" for k, v in properties.items())
    if property_str:
        signature = f"{node1_signature} -[{relationship.type}({property_str})]-> {node2_signature}"
    else:
        signature = f"{node1_signature} -[{relationship.type}]-> {node2_signature}"
    return signature

def get_node_name(node: Node) -> str:
    return node.properties["name"]
def get_node_id(node: Node) -> str:
    return node.id

#### 构造输入decoder-only模型的文本，获取所有层的hidden states以及k向量。
* 此处返回值是：
    1. all_layers_hidden_states: List[Tensor], 每个元素的shape都是(hidden_size)，hidden_size是模型的隐藏层维度3584，层数是29 = (embedding layer + 28 transformer layers)
    2. all_layers_k: List[Tensor], 每个元素的shape都是(k_dim)，k_dim是模型中Attention中K向量的维度512，层数是28（不存在embedding layer）
    3. 当aim_word在输入文本中多次出现时，选择的位置是最后一次出现的位置，即sample_prompt2中""{word}""的位置, 若 aim_word 在分词后被分成多个token，则会将这些token对应层的向量各自进行平均，得到一个hidden_size维度的向量作为该位置的向量表示。（也可以选择拼接这些token对应层的向量，得到一个hidden_size * num_tokens维度的向量作为该位置的向量表示，但这样将造成维度爆炸，同时如果Node 的num_tokens不同，Node间将无法进行相似度计算）
    3. 关于模型内部计算的相关代码见`server.py`
* Qwen2.5模型架构：
    1. embedding layer: 1层，输出维度hidden_size=3584
    2. transformer layers: 28层 每层：
        * self-attention: Q为3584维，K、V为512维，每个头为128维，因此Q有28个头，K、V有4个头，每个K，V的 head 与 7 个Q的 head 进行attention计算。
        * mlp: 先升维到4倍hidden_size，即14336维，再降维回hidden_size=3584维
        * RMSNorm层: 计算hidden states的RMS值，对hidden states进行归一化
* 因此 all_layers_hidden_states的实际是每个transformer layer的输出，即经过self-attention、mlp、RMSNorm层计算后的hidden states
* all_layers_k的实际是每个transformer layer中self-attention计算得到的K向量，量是经过线性变换后的K向量，维度是512维

In [ ]:
def get_all_vectors(node_id:str, relations:list[Relationship]) -> List[Tensor]:
    if len(relations) > 15:
        import random
        relations = random.sample(relations, 15)
    rel_signs = [get_relationship_signature(rel) for rel in relations]
    rel_signs_str = ";".join(rel_signs)
    aim_node:Node = relations[0].source if relations[0].source.id == node_id else relations[0].target

    assert aim_node.id == node_id, f"aim_node.id {aim_node.id} != node_id {node_id}"

    aim_word = get_node_name(aim_node)
    text = sample_prompt1.format(word=aim_word, relations=rel_signs_str)

    all_layers_hidden_states = get_all_hidden_states(text, aim_word)
    return all_layers_hidden_states

def get_all_vectors_k(node_id:str, relations:list[Relationship]) -> List[Tensor]:
    if len(relations) > 15:
        import random
        relations = random.sample(relations, 15)
    rel_signs = [get_relationship_signature(rel) for rel in relations]
    rel_signs_str = ";".join(rel_signs)
    aim_node:Node = relations[0].source if relations[0].source.id == node_id else relations[0].target

    assert aim_node.id == node_id, f"aim_node.id {aim_node.id} != node_id {node_id}"

    aim_word = get_node_name(aim_node)
    text = sample_prompt1.format(word=aim_word, relations=rel_signs_str)

    all_layers_k = get_all_k_vectors(text, aim_word)
    return all_layers_k

#### 除了计算得到hidden states和k向量之外，我们探索的其他方式还有：
1. last_layer_last_token: 
* 直接使用最后一层的hidden states中最后一个token的向量作为节点的表示(利用decoder-only模型的自回归特性，最后一个token的hidden states即模型将要输出的下一个token的向量)
2. topk_attentioned_v: 
* 对Attention中的V向量，如果直接按照attention分数矩阵(归一化后的 K dot Q.T)加权求和，就是原始的attention输出O
* 现在：不用注意力分数矩阵的全部，每个位置我只取top_k个注意力分数值最高的，只用这top_k个分数对v进行加权计算，得到这个输出，记为attentioned_v。这个向量可以看作是模型在计算attention时，最关注的v向量的加权和。
* 这里同样也是选择若干层的attentioned_v向量进行聚合，得到最终的节点表示。
* 这里面其实也可以是K向量，因为现在大部分模型的K、V向量是完全一样的，即 k_proj 和 v_proj的权重矩阵是一样的
* 这里面对于top_k的选择实验时取了 len(rel) + 3，即关系数量加3（因为除了关系之外，还有一些特殊token也可能被模型关注，比如[CLS]、[SEP]等）。
3. sentence_embedding_instructed:
* 数据的处理方式基本相同，只是后遭prompt时是：Instruction + Query, 不会去拆模型而是直接用最后的输出向量作为节点的表示。

In [ ]:
def get_top_k_attned_v(node_id:list[str], relations:list[list[Relationship]]) -> List[Tensor]: # 这里使用批量计算
    
    if len(node_id) != len(relations):
        raise ValueError("node_id和relations的长度必须相同")
    texts = []
    k_s = []
    aim_words = []
    for node_id_i, relations_i in zip(node_id, relations):
        if len(relations_i) > 10:
            import random
            relations_i = random.sample(population=relations_i, k=10)
        k = len(relations_i) + 3
        rel_signs = [get_relationship_signature(rel) for rel in relations_i]
        rel_signs_str = ";".join(rel_signs)
        aim_node:Node = relations_i[0].source if relations_i[0].source.id == node_id_i else relations_i[0].target

        assert aim_node.id == node_id_i, f"aim_node.id {aim_node.id} != node_id_i {node_id_i}"

        aim_word = get_node_name(aim_node)
        text = sample_prompt2.format(word=aim_word, type=aim_node.type if aim_node.type != "unknown" else "Entity", relations=rel_signs_str)
        texts.append(text)
        aim_words.append(aim_word)
        k_s.append(k)
    
    top_k_attned_v = get_top_k_attned_v(texts, aim_words, layer_idx, "mean", k=k_s)
    return top_k_attned_v

In [ ]:
# all_layers_hidden_states/all_layers_k/top_k_attned_v/last_token
kg_type = "yago"
with open(f"{input_data_dir}/{kg_type}_selected_rels.pkl", "rb") as f:
    node2rels = pickle.load(f)      # 原始的node_id 到关系列表的映射，关系列表中的每个元素是一个Relationship对象
with open(f"{input_data_dir}/{kg_type}_node2rels.pkl", "rb") as f:
    node2all_rels = pickle.load(f)  # 经过挑选算法挑选出的node_id到关系列表的映射

ans = {} # {id:all_layers_hidden_states}
import tqdm
for node_id, relations in tqdm.tqdm(node2rels.items()):
    if len(relations) == 0:
        relations = node2all_rels.get(node_id, [])
    all_layers_hidden_states = get_all_vectors(node_id, relations)
    ans[node_id] = all_layers_hidden_states
with open(f"{out_data_dir}/{kg_type}_all_layers_hidden_states.pkl", "wb") as f:
    pickle.dump(ans, f)
"""
ans = {
    node_id: List[Tensor] # all_layers_hidden_states / all_layers_k / top_k_attned_v / last_token
}
"""

#### 从 all_layers_hidden_states / all_layers_k 中聚合目标层的向量
#### 聚合方式：平均池化（mean pooling）
#### 选择目标层:layer_idx = [x, xx, ...], 返回将这些层的向量进行平均池化后的结果 shape：(hidden_size) 或 (k_dim)

In [ ]:
# 从计算的all_layers_k_vectors中聚合目标层的向量
import torch
def aggregate_target_layer_k_vectors(kg_type:str, layer_idx:list[int], idx_num):
    with open(f"{out_data_dir}/{kg_type}_all_layers_k_vectors.pkl", "rb") as f:
        all_layers_hidden_states = pickle.load(f)
    
    target_layer_vectors = {}
    for node_id, layers_hidden_states in tqdm.tqdm(all_layers_hidden_states.items()):
        # layers_hidden_states的结构是一个list，长度为层数，每个元素是一个Tensor，形状为[hidden_size]
        selected_layers = [layers_hidden_states[i] for i in layer_idx]
        # 将选定层的向量进行聚合，这里使用平均聚合
        aggregated_vector = torch.mean(torch.stack(selected_layers), dim=0)
        target_layer_vectors[node_id] = aggregated_vector 
    
    with open(f"{out_data_dir}/{kg_type}_aim_word_vectors_k(sample_1)(layer_idx{idx_num}).pkl", "wb") as f:
        pickle.dump(target_layer_vectors, f)

# 从计算的all_layers_hidden_states中聚合目标层的向量
def aggregate_target_layer_vectors(kg_type:str, layer_idx:list[int], idx_num):
    with open(f"{out_data_dir}/{kg_type}_all_layers_hidden_states.pkl", "rb") as f:
        all_layers_hidden_states = pickle.load(f)
    
    target_layer_vectors = {}
    for node_id, layers_hidden_states in tqdm.tqdm(all_layers_hidden_states.items()):
        # layers_hidden_states的结构是一个list，长度为层数，每个元素是一个Tensor，形状为[hidden_size]
        selected_layers = [layers_hidden_states[i] for i in layer_idx]
        # 将选定层的向量进行聚合，这里使用平均聚合
        aggregated_vector = torch.mean(torch.stack(selected_layers), dim=0)
        target_layer_vectors[node_id] = aggregated_vector 
    
    with open(f"{out_data_dir}/{kg_type}_aim_word_vectors(sample_1)(layer_idx{idx_num}).pkl", "wb") as f:
        pickle.dump(target_layer_vectors, f)

#### 用所选目标层聚合得到的向量计算相似度
#### `matched_nodes_id_to_node: Dict [node_id, List[node_id] ]`, 即preprocess阶段由聚类结果得到的，在匹配到的节点中，每个节点与它相似度最高的簇中的节点的id列表
* 为每个node_id，计算它与matched_nodes_id_to_node[node_id]中每个节点的相似度

In [ ]:
kg_type1 = "icews"
kg_type2 = "yago"
emb_type = "aim_word_vectors(sample_1)(layer_idx3)" # 选择emb的类型，hidden_states / k_vectors，sample_1/sample_2, layer_idx_k
with open(f"{input_data_dir}/{kg_type1}_matched_nodes_id_to_nodes(qwen).pkl", "rb") as f:
    matched_nodes_id_to_node = pickle.load(f)
print(f"Total matched nodes: {len(matched_nodes_id_to_node)}")
"""
{
    icews_node_id1: set(yago_node_id1, yago_node_id2,...),
    icews_node_id2: set(yago_node_id3, yago_node_id4,...),
     ...
}
"""
with open(f"{out_data_dir}/{kg_type1}_{emb_type}.pkl", "rb") as f:
    kg1_node_id_to_vector = pickle.load(f)
with open(f"{out_data_dir}/{kg_type2}_{emb_type}.pkl", "rb") as f:
    kg2_node_id_to_vector = pickle.load(f)
# 将所以matched的节点与set中的节点计算相似度，并按相似度排序，保存为一个新的字典
results = []
"""
[
    {
        "node_1": icews_node_id,
        "node_2": wiki_node_id,
        "sim_{emb_type}": sim_value
    }
]
"""
import tqdm
for node_id1, matched_node_ids2 in tqdm.tqdm(matched_nodes_id_to_node.items()):
    vector1 = kg1_node_id_to_vector.get(node_id1)
    if vector1 is None:
        print(f"Node {node_id1} in {kg_type1} has no vector, skip.")
        continue
    for node_id2 in matched_node_ids2:
        vector2 = kg2_node_id_to_vector.get(node_id2)
        if vector2 is None:
            print(f"Node {node_id2} in {kg_type2} has no vector, skip.")
            continue
        sim = F.cosine_similarity(vector1.unsqueeze(0), vector2.unsqueeze(0)).item()
        results.append({
            "node_1": node_id1,
            "node_2": node_id2,
            f"sim_{emb_type}": sim
        })
print(f"Finish the type:{emb_type}")
with open(f"{out_data_dir}/matched_nodes_similarity_{emb_type}.json", "w") as f:
    json.dump(results, f, indent=4)

#### 利用计算得到的相似度分析Hits@k

In [ ]:
with open(f"{out_data_dir}/matched_nodes_similarity_{emb_type}.json", "r") as f:
    results = json.load(f)
graph_dir = "../../data/source_data/icews_yago"
with open(f"{graph_dir}/icews2yago.json", "r") as f:
    icews_id2yago_id = json.load(f)
# 计算Hits@K等指标
# 首先将results按照node_1进行分组，然后对于每个node_1，按照sim_{emb_type}从大到小排序，看看node_2与node_1是否匹配，并计算Hits@K等指标
from collections import defaultdict
grouped_results = defaultdict(list)

for item in tqdm.tqdm(results, desc="Processing results"):
    grouped_results[item["node_1"]].append(item)
# 计算Hits@K
hits_at_k = {1: 0, 3: 0, 5: 0, 10: 0}
total = 0

for node_1, items in tqdm.tqdm(grouped_results.items(), desc="Processing grouped results"):
    total += 1
    # 按照sim_{emb_type}从大到小排序
    items.sort(key=lambda x: x[f"sim_{emb_type}"], reverse=True)
    # 找到node_2与node_1相同的项，看看它的排名位置
    # 对于node_1，要使用icews_id2yago_id映射到对应的yago节点id，然后再与node_2进行比较
    matched_id = icews_id2yago_id.get(node_1)
    if matched_id is None:
        print(f"Warning: node_id {node_1} not found in icews_id2yago_id mapping.")
    for rank, item in enumerate(items, start=1): # rank从1开始，rank=1表示相似度最高的节点
        if item["node_2"] == matched_id:
            if rank <= 1:
                hits_at_k[1] += 1
            if rank <= 3:
                hits_at_k[3] += 1
            if rank <= 5:
                hits_at_k[5] += 1
            if rank <= 10:
                hits_at_k[10] += 1
            break
print(f"Finish calculating Hits@K for type: {emb_type}")
print(f"# {emb_type} ")
print(f"# layer_idx: {layer_idx}")
print("\"\"\"")
print(f"Total matched nodes: {total}")
for k in hits_at_k:
    print(f"Hits@{k}: {hits_at_k[k]} ({hits_at_k[k]/total:.4f})")
print("\"\"\"")